In [10]:
# Perform imports
import pandas as pd


# Create a DataFrame from the .csv file and # remove unuse columns
artikel = pd.read_csv("isjd8juni.csv", names=['id_artikel', 'id_direktori', 'key', 'month', 'number', 'halaman', 'title', 'volume', 'year', 'deskriptor', 'abstract', 'sari', 'id_category', 'subject'])
artikel.drop(columns =['deskriptor','key', 'month', 'subject', 'number', 'halaman', 'volume', 'abstract', 'sari'], axis=1, inplace = True)
artikel.sort_values(by=['id_category'], inplace=True)
artikel = artikel.drop_duplicates(['title'], keep='last') # remove duplicate rows
artikel.fillna(0,inplace=True) # replace empty values with 0
artikel['year'] = artikel['year'].astype('int64') # change column data type
artikel = artikel[(artikel['year'] >= 2017)]

category = pd.read_csv("category_aji.csv", names=['id_category', 'name_cat', 'no_kelas', 'view'])
category.drop(columns =['no_kelas'], axis=1, inplace = True)
category.replace(to_replace = "Program Komputer dan Teknologi Informasi", value = "Komputer dan Teknologi Informasi", inplace = True)
author = pd.read_csv("masterauthor_baru_ai.csv", names=['author_id', 'name', 'alamat', 'email', 'instansi', 'no_tlp', 'crdt', 'crby', 'updt', 'upby'], dtype={'updt': object, 'upby': object})
author.drop(columns =['alamat', 'email', 'no_tlp', 'crdt', 'crby', 'updt', 'upby'], axis=1, inplace = True)
author['instansi'] = author['instansi'].str.strip() # remove white space in instansi column
author['name'] = author['name'].str.strip() # remove white space in instansi column
authorbaru = pd.read_csv("author_baru_ai.csv", names=['jurnal_author_id', 'author_id', 'author_ke', 'id_artikel', 'crdt', 'crby', 'updt', 'upby'], dtype={'author_id':int, 'crdt': object,  'crby': object, 'updt': object, 'upby': object, 'author_ke': object})
authorbaru.drop(columns =['jurnal_author_id','crdt', 'crby', 'updt', 'upby'], axis=1, inplace = True)
direktori = pd.read_csv("direktori_aji.csv", names=['id_direktori', 'id_user', 'id_category', 'tanggal', 'bahasa', 'issn', 'eissn', 'judul_jurnal', 'judulsebelumnya', 'judulbaru', 'judulpararel', 'editor', 'penerbit', 'alamat', 'tahun_jurnal', 'tiras', 'harga', 'catatan', 'li', 'frek', 'ci', 'mb', 'biaya', 'akreditasi', 'abstrak', 'kp', 'subject', 'deskind', 'link', 'waktu', 'url','key', 'up', 'pass', 'crdt', 'crby', 'updt', 'upby', 'ket', 'idins', 'jmb', 'sp', 'nomor_akreditasi', 'idpar', 'isac', 'idanal', 'tglanal', 'idval', 'tglval'])
direktori.drop(columns =['akreditasi','nomor_akreditasi','subject','id_user','tanggal', 'bahasa', 'issn', 'eissn', 'judulsebelumnya', 'judulbaru', 'judulpararel', 'editor', 'alamat', 'tiras', 'harga', 'catatan', 'li', 'frek', 'ci', 'mb', 'biaya', 'abstrak', 'kp', 'deskind', 'link', 'waktu', 'url','key', 'up', 'pass', 'crdt', 'crby', 'updt', 'upby', 'ket', 'idins', 'jmb', 'sp', 'idpar', 'isac', 'idanal', 'tglanal', 'idval', 'tglval'], axis=1, inplace = True)
direktori.fillna(0,inplace=True)
direktori['tahun_jurnal'] = direktori['tahun_jurnal'].astype('int64')

# Merge table
artikelKategori= pd.merge(left=artikel,right=category, left_on='id_category', right_on='id_category') # artikel with category
penulis= pd.merge(left=author,right=authorbaru, left_on='author_id', right_on='author_id') # author with instansi
penulis = penulis[(penulis['instansi'].notnull()) & (penulis['instansi'] != "-") ] # remove null and zero
isjd= pd.merge(left=artikelKategori,right=penulis, left_on='id_artikel', right_on='id_artikel') # artikel with author
isjd.drop(columns = ['id_category','view'], axis=1, inplace = True)
artikelKategori.drop(columns = ['id_direktori','id_category','view'], axis=1, inplace = True)
jurnal = pd.merge(left=direktori,right=category, left_on='id_category', right_on='id_category') # direktori with category
jurnal.drop(columns = ['id_category', 'view'], axis=1, inplace = True)
isjd.fillna(0,inplace=True)  #replace empty values with 0
artikelKategori.fillna(0,inplace=True)  #replace empty values with 0
jurnal.fillna(0,inplace=True)  #replace empty values with 0
artikelJurnal= pd.merge(left=jurnal,right=isjd, left_on='id_direktori', right_on='id_direktori') # artikel with category
artikelJurnal.drop(columns = ['id_direktori','id_artikel','title','name_cat_y','author_id','name','author_ke'], axis=1, inplace = True)
#isjdForAuthor = 
artikelJurnal2= pd.merge(left=artikel,right=jurnal, left_on='id_direktori', right_on='id_direktori') # Original article with directory 
artikelJurnal2.drop(columns = ['id_direktori','title','id_category','tahun_jurnal'], axis=1, inplace = True)


# Create a DataFrame for artikel category, year, institution chart
isjdFeatures = isjd.groupby(['instansi','name_cat','year'], as_index=False).count().sort_values('id_artikel', ascending=False)
isjdInstansi = isjd.groupby(['instansi'], as_index=False).count().sort_values('id_artikel', ascending=False)
isjdYear = artikelKategori.groupby(['year'], as_index=False).count().sort_values('year', ascending=False)
isjdYearInstansi = isjd.groupby(['year', 'instansi'], as_index=False).count().sort_values('year', ascending=True)
isjdCategoryYear = artikelKategori.groupby(['year','name_cat'], as_index=False).count().sort_values('year', ascending=False)
allYear = isjdCategoryYear.groupby(['name_cat'], as_index=False).sum().sort_values('id_artikel', ascending=False)
isjdCategory = artikelKategori.groupby(['name_cat'], as_index=False).count().sort_values('id_artikel', ascending=False)
isjdCategoryInstansi = isjd.groupby(['instansi','name_cat'], as_index=False).count().sort_values('id_artikel', ascending=False)
isjdInstansiCategoryYear = isjd.groupby(['instansi','name_cat', 'year'], as_index=False).count().sort_values('year', ascending=False)
artikelJurnalCategoryYearInstansi = artikelJurnal.groupby(['judul_jurnal','penerbit','name_cat_x', 'year','instansi'], as_index=False).count().sort_values('tahun_jurnal', ascending=False)
artikelJurnalCategoryYearInstansi.rename(columns={'judul_jurnal': 'Nama Jurnal', 'penerbit': 'Penerbit', 'name_cat_x':'Kategori','year':'Tahun','instansi':'Institusi' ,'tahun_jurnal':'Jumlah Artikel'}, inplace=True)
artikelJurnalCategoryYear = artikelJurnal2.groupby(['judul_jurnal','penerbit','name_cat', 'year'], as_index=False).count().sort_values('id_artikel', ascending=False)
artikelJurnalCategoryYear.rename(columns={'judul_jurnal': 'Nama Jurnal', 'penerbit': 'Penerbit', 'name_cat':'Kategori','year':'Tahun','id_artikel':'Jumlah Artikel'}, inplace=True)


# Create a DataFrame for author chart
Penulis = isjd.groupby(['author_id','name','instansi','year'], as_index=False).count().sort_values('year', ascending=False)
Penulis["Nama dan ID Penulis"] = Penulis["name"] + " " + "(" + Penulis["author_id"].astype(str) + ")"
categoryPenulis = isjd.groupby(['author_id','name', 'instansi', 'name_cat'], as_index=False).count().sort_values('id_artikel', ascending=False)
categoryPenulis["Nama dan ID Penulis"] = categoryPenulis["name"] + " " + "(" + categoryPenulis["author_id"].astype(str) + ")"
isjdForAuthor= pd.merge(left=isjd,right=jurnal, left_on='id_direktori', right_on='id_direktori') # Original article with directory 
isjdForAuthor.drop(columns = ['name_cat_x','judul_jurnal','penerbit','tahun_jurnal'], axis=1, inplace = True)
isjdForAuthor.rename(columns={'name_cat_y':'name_cat'}, inplace=True)

# Create a DataFrame for author table
isjdPenulis = isjd.groupby(['author_id','name', 'instansi'], as_index=False).count().sort_values('id_artikel', ascending=False)
isjdPenulis["Nama dan ID Penulis"] = isjdPenulis["name"] + " " + "(" + isjdPenulis["author_id"].astype(str) + ")"
isjdPenulis.rename(columns={'instansi':'Institusi','name_cat':'Kategori', 'year':'Tahun' ,'id_artikel':'Jumlah Artikel'}, inplace=True)
isjdPenulisSelect = isjdPenulis[['Nama dan ID Penulis', 'Institusi','Kategori','Tahun','Jumlah Artikel']]
isjdPenulis = isjdPenulis[['Nama dan ID Penulis', 'Institusi','Jumlah Artikel']]
isjdTopPenulis = isjdPenulis.nlargest(10,'Jumlah Artikel', keep='first')

isjdPenulisCategory = isjdForAuthor.groupby(['author_id','name', 'instansi', 'name_cat'], as_index=False).count().sort_values('id_artikel', ascending=False)
isjdPenulisCategory["Nama dan ID Penulis"] = isjdPenulisCategory["name"] + " " + "(" + isjdPenulisCategory["author_id"].astype(str) + ")"
isjdPenulisCategory = isjdPenulisCategory[['Nama dan ID Penulis', 'instansi','name_cat','id_artikel']]
isjdPenulisCategory.rename(columns={'instansi':'Institusi','name_cat':'Kategori','id_artikel':'Jumlah Artikel'}, inplace=True)

isjdPenulisYearCategory = isjdForAuthor.groupby(['author_id','name', 'instansi', 'year', 'name_cat'], as_index=False).count().sort_values('id_artikel', ascending=False)
isjdPenulisYearCategory["Nama dan ID Penulis"] = isjdPenulisYearCategory["name"] + " " + "(" + isjdPenulisYearCategory["author_id"].astype(str) + ")"
isjdPenulisYearCategory = isjdPenulisYearCategory[['Nama dan ID Penulis', 'instansi', 'year', 'name_cat', 'id_artikel']]
isjdPenulisYearCategory.rename(columns={'instansi':'Institusi','name_cat':'Kategori','year':'Tahun' ,'id_artikel':'Jumlah Artikel'}, inplace=True)


# Create a DataFrame for jurnal chart
jurnalYear = jurnal.groupby(['tahun_jurnal'], as_index=False).count().sort_values('tahun_jurnal', ascending=False)
jurnalCategory = jurnal.groupby(['name_cat'], as_index=False).count().sort_values('id_direktori', ascending=False)
jurnalCategoryYear = jurnal.groupby(['tahun_jurnal', 'name_cat'], as_index=False).count().sort_values('tahun_jurnal', ascending=False)


#create a Dataframe for instansi & author Collaboration
collabIsjd = isjd[isjd['author_ke'] != '1']

collabIsjdId = collabIsjd.drop(collabIsjd.iloc[:, 1:8], axis = 1)
collabIsjdId.drop_duplicates(subset ="id_artikel", keep = 'first', inplace = True) 

df1 = pd.merge(left=isjd,right=collabIsjdId, left_on='id_artikel', right_on='id_artikel') 
df1.groupby('id_artikel').agg(lambda x: set(x)).reset_index()

collabAuthor = pd.merge(df1, df1, on=['id_artikel'])
collabAuthor = collabAuthor[~(collabAuthor['name_x'] == collabAuthor['name_y'])]
collabAuthor["Nama dan ID Penulis"] = collabAuthor["name_x"] + " " + "(" + collabAuthor["author_id_x"].astype(str) + ")"

collabInstansi = collabAuthor.groupby(['id_artikel', 'instansi_x','instansi_y','year_x','name_cat_x'], as_index=False).count().sort_values('id_artikel', ascending=False)
collabInstansi.rename(columns={'id_artikel': 'Jumlah Kolaborasi', 'instansi_x': 'Institusi', 'instansi_y':'Kolaborator','year_x':'Tahun' ,'name_cat_x':'Kategori'}, inplace=True)

collabInstansicategoryYear = collabInstansi.groupby(['Institusi','Kolaborator','Kategori','Tahun'],as_index=False).count().sort_values('Jumlah Kolaborasi', ascending=False)
collabInstansicategoryYear = collabInstansicategoryYear[['Institusi','Kolaborator','Kategori','Tahun','Jumlah Kolaborasi']]

collabInstansi1 = collabInstansi.groupby(['Institusi'],as_index=False).count().sort_values('Jumlah Kolaborasi', ascending=False)
collabInstansi1 = collabInstansi1[['Institusi', 'Jumlah Kolaborasi']]

isjdTopKolaborasi = collabInstansi1.nlargest(10,'Jumlah Kolaborasi', keep='first')

collabInstansiKolaborator = collabInstansi.groupby(['Institusi','Kolaborator'],as_index=False).count().sort_values('Jumlah Kolaborasi', ascending=False)
collabInstansiKolaborator = collabInstansiKolaborator[['Institusi','Kolaborator','Jumlah Kolaborasi']]
collabInstansiCategory = collabInstansi.groupby(['Institusi','Kolaborator','Kategori'],as_index=False).count().sort_values('Jumlah Kolaborasi', ascending=False)
collabInstansiCategory = collabInstansiCategory[['Institusi','Kolaborator','Kategori','Jumlah Kolaborasi']]
collabcategoryYear = collabInstansi.groupby(['Kategori','Tahun'],as_index=False).count().sort_values('Jumlah Kolaborasi', ascending=False)
collabcategoryYear = collabcategoryYear[['Kategori','Tahun','Jumlah Kolaborasi']]
collabInstansicategoryYear2 = collabInstansi.groupby(['Institusi','Kategori','Tahun'],as_index=False).count().sort_values('Tahun', ascending=False)
collabInstansicategoryYear2 = collabInstansicategoryYear2[['Institusi','Kategori','Tahun','Jumlah Kolaborasi']]
collabInstansiYear = collabInstansi.groupby(['Institusi','Tahun'],as_index=False).count().sort_values('Tahun', ascending=False)
collabInstansiYear =  collabInstansiYear[['Institusi','Tahun','Jumlah Kolaborasi']]

collabAuthor1 = collabAuthor.groupby(['year_x','name_cat_x','author_id_x','name_x','instansi_x','name_y','instansi_y','Nama dan ID Penulis'], as_index=False).count().sort_values('id_artikel', ascending=False)
collabAuthor1.rename(columns={'year_x': 'Tahun','name_cat_x': 'Kategori','author_id_x': 'ID Penulis','name_x': 'Nama','instansi_x':'Institusi','name_y':'Kolaborator','instansi_y':'Institusi Kolaborator','id_artikel': 'Jumlah Kolaborasi'}, inplace= True)

authorTopKolaborasi = collabAuthor1.groupby(['Nama dan ID Penulis','Institusi'],as_index=False).sum().sort_values('Jumlah Kolaborasi', ascending=False)
#authorTopKolaborasi["Nama dan Instansi"] =  authorTopKolaborasi["Nama"] + " " + "(" + authorTopKolaborasi["Institusi"] + ")"
authorTopKolaborasi = authorTopKolaborasi.head(10)

isjdAuthorCollabCategory = collabAuthor1.groupby(['Kategori','Tahun','Institusi'],as_index=False).sum().sort_values('Jumlah Kolaborasi', ascending=False)
isjdAuthorCollabTahun = collabAuthor1.groupby(['Nama dan ID Penulis','Institusi','Kolaborator','Institusi Kolaborator','Tahun'],as_index=False).sum().sort_values('Jumlah Kolaborasi', ascending=False)

collabAuthor.rename(columns={'year_x': 'Tahun','name_cat_x': 'Kategori','author_id_x': 'ID Penulis','name_x': 'Nama','instansi_x':'Institusi','name_y':'Kolaborator','instansi_y':'Institusi Kolaborator','id_artikel': 'ID Artikel'}, inplace= True)
collabAuthor2 = collabAuthor.groupby(['Tahun','ID Artikel','Nama dan ID Penulis','Institusi'],as_index=False).count().sort_values('Tahun', ascending=False) #Merge ID Artikel First to get count of article publiched with collaboration
collabAuthor2 = collabAuthor2.groupby(['Tahun','Nama dan ID Penulis','Institusi'],as_index=False).count().sort_values('Tahun', ascending=False)

                                        
# Remove unuse coloumn
isjdFeatures = isjdFeatures[['instansi','name_cat','year']]
isjdInstansi = isjdInstansi[['instansi', 'id_artikel']]
isjdCategory = isjdCategory[['name_cat', 'id_artikel']]
isjdYear = isjdYear[['year', 'id_artikel']]
isjdInstansiCategoryYear = isjdInstansiCategoryYear[['instansi','name_cat','year', 'id_artikel']]
isjdCategoryYear = isjdCategoryYear[['year', 'name_cat', 'id_artikel']]
isjdYearInstansi = isjdYearInstansi[['year', 'instansi', 'id_artikel']]
isjdCategoryInstansi = isjdCategoryInstansi[['instansi', 'name_cat','id_artikel']]
Penulis = Penulis[['Nama dan ID Penulis', 'instansi', 'year','id_artikel']]
jurnalYear = jurnalYear[['tahun_jurnal', 'id_direktori']]
jurnalCategory = jurnalCategory[['name_cat', 'id_direktori']]
jurnalCategoryYear = jurnalCategoryYear[['tahun_jurnal', 'name_cat','id_direktori']]
#collabAuthor = collabAuthor[['id_artikel','year_x','name_cat_x','author_id_x','name_x','instansi_x','name_y','instansi_y','Nama dan ID Penulis']]
authorTopKolaborasi = authorTopKolaborasi[['Nama dan ID Penulis','Institusi','Jumlah Kolaborasi']]
collabAuthor2 = collabAuthor2[['Tahun','Nama dan ID Penulis','Institusi', 'ID Artikel']]
isjdAuthorCollabTahun = isjdAuthorCollabTahun[['Nama dan ID Penulis','Institusi','Kolaborator','Institusi Kolaborator','Tahun','Jumlah Kolaborasi']]
isjdAuthorCollabCategory = isjdAuthorCollabCategory[['Kategori','Tahun','Institusi','Jumlah Kolaborasi']]


#export to CSV
isjdFeatures.to_csv('isjdFeatures.csv', header=False, index=False)
isjdInstansi.to_csv('isjdInstansi.csv', header=False, index=False)
isjdCategory.to_csv('isjdCategory.csv', header=False, index=False)
isjdTopPenulis.to_csv('isjdTopPenulis.csv', header=False, index=False)
isjdYear.to_csv('isjdYear.csv', header=False, index=False)
isjdInstansiCategoryYear.to_csv('isjdInstansiCategoryYear.csv', header=False, index=False)
isjdCategoryYear.to_csv('isjdCategoryYear.csv', header=False, index=False)
isjdYearInstansi.to_csv('isjdYearInstansi.csv', header=False, index=False)
isjdCategoryInstansi.to_csv('isjdCategoryInstansi.csv', header=False, index=False)
isjdPenulisSelect.to_csv('isjdPenulisSelect.csv', header=False, index=False)
isjdPenulisCategory.to_csv('isjdPenulisCategory.csv', header=False, index=False)
isjdPenulisYearCategory.to_csv('isjdPenulisYearCategory.csv', header=False, index=False)
Penulis.to_csv('Penulis.csv', header=False, index=False)
jurnalYear.to_csv('jurnalYear.csv', header=False, index=False)
jurnalCategory.to_csv('jurnalCategory.csv', header=False, index=False)
jurnalCategoryYear.to_csv('jurnalCategoryYear.csv', header=False, index=False)
collabInstansi1.to_csv('collabInstansi.csv', header=False, index=False)
collabInstansicategoryYear.to_csv('collabInstansicategoryYear.csv', header=False, index=False)
isjdTopKolaborasi.to_csv('isjdTopKolaborasi.csv', header=False, index=False)
collabInstansiKolaborator.to_csv('collabInstansiKolaborator.csv', header=False, index=False)
collabInstansiCategory.to_csv('collabInstansiCategory.csv', header=False, index=False)
collabcategoryYear.to_csv('collabcategoryYear.csv', header=False, index=False)
collabInstansicategoryYear2.to_csv('collabInstansicategoryYear2.csv', header=False, index=False)
collabInstansiYear.to_csv('collabInstansiYear.csv', header=False, index=False)
authorTopKolaborasi.to_csv('authorTopKolaborasi.csv', header=False, index=False)
collabAuthor2.to_csv('collabAuthor2.csv', header=False, index=False)
isjdAuthorCollabTahun.to_csv('isjdAuthorCollabTahun.csv', header=False, index=False)
isjdAuthorCollabCategory.to_csv('isjdAuthorCollabCategory.csv', header=False, index=False)
artikelJurnalCategoryYearInstansi.to_csv('artikelJurnalCategoryYearInstansi.csv', header=False, index=False)
artikelJurnalCategoryYear.to_csv('artikelJurnalCategoryYear.csv', header=False, index=False


'\ncategory = pd.read_csv("category_aji.csv", names=[\'id_category\', \'name_cat\', \'no_kelas\', \'view\'])\ncategory.drop(columns =[\'no_kelas\'], axis=1, inplace = True)\ncategory.replace(to_replace = "Program Komputer dan Teknologi Informasi", value = "Komputer dan Teknologi Informasi", inplace = True)\nauthor = pd.read_csv("masterauthor_baru_ai.csv", names=[\'author_id\', \'name\', \'alamat\', \'email\', \'instansi\', \'no_tlp\', \'crdt\', \'crby\', \'updt\', \'upby\'], dtype={\'updt\': object, \'upby\': object})\nauthor.drop(columns =[\'alamat\', \'email\', \'no_tlp\', \'crdt\', \'crby\', \'updt\', \'upby\'], axis=1, inplace = True)\nauthor[\'instansi\'] = author[\'instansi\'].str.strip() # remove white space in instansi column\nauthor[\'name\'] = author[\'name\'].str.strip() # remove white space in instansi column\nauthorbaru = pd.read_csv("author_baru_ai.csv", names=[\'jurnal_author_id\', \'author_id\', \'author_ke\', \'id_artikel\', \'crdt\', \'crby\', \'updt\', \'upby\'], dty

In [6]:
x = collabAuthor2.groupby(['Nama dan ID Penulis','Institusi'],as_index=False).sum().sort_values('ID Artikel', ascending=False)
x

,Nama dan ID Penulis,Institusi,Tahun,ID Artikel
26010,Sutardi (85159),Universitas Halu Oleo,4035,20
14993,Livana PH (423968),STIKES Kendal,4035,19
6615,Djatmika (151239),Universitas Sebelas Maret Surakarta,4035,18
13662,Jumadil Nangi (394824),Universitas Halu Oleo,6054,14
17144,Muh. Yamin (172094),Universitas Halu Oleo,4035,13
...,...,...,...,...
11362,Hrry Koswara (429031),STKS Bandung,2018,1
11361,Hr.Cahyo Diartho (440513),Pilih Institusi,2019,1
11360,Hotnier Sipahutar (395358),"Research and Development Agency, Ministry of Home",2018,1
11358,Hotnida Sitorus (166796),Balai Penelitian dan Pengembangan Kesehatan Ba...,2018,1


In [2]:
isjd[isjd['name'] == 'Sutardi']

,id_artikel,id_direktori,title,year,name_cat,author_id,name,instansi,author_ke
74,11197703,102654.0,Hukum mengucapkan selamat natal menurut pandan...,2018,Agama,444808,Nurhayati,Unknown,1
83,11197691,102653.0,Sistim dan prinsip distribusi hasil usaha dala...,2018,Agama,444809,Nurhayati,Unknown,1
3620,11137555,1123.0,Komposisi nutrisi rumput laut calcareous halim...,2017,Biologi,444676,Nurhayati,Kementerian Kelautan dan Perikanan,1
5880,11203350,186.0,Pelaksanaan tindak pidana perkosaan: studi kom...,2018,Hukum,442685,Nurhayati,UIN Sumatera Utara,1
8129,11216162,110582.0,Aktivitas fisik dan kadar kolesterol total den...,2018,Kesehatan dan Kedokteran,442683,Nurhayati,STIKES Widya Nusantara Palu,1
10803,11224896,110825.0,Effectiveness of Jasmine Oil (<i>Jasminum Offi...,2018,Kesehatan dan Kedokteran,444725,Nurhayati,Universitas Aisyah Pringsewu,2
10937,11225932,110825.0,Iron deficiency anemia and current state of kn...,2019,Kesehatan dan Kedokteran,442682,Nurhayati,Akademi Keperawatan Baitul Hikmah Bandar Lampung,1
17128,11225912,12830.0,Kejadian kanker payudara (studi retrospektif)...,2019,Manajemen,442682,Nurhayati,Akademi Keperawatan Baitul Hikmah Bandar Lampung,1
18402,11206886,101162.0,Kajian uji konfrontasi terhadap bakteri pathog...,2017,Manajemen,444683,Nurhayati,Politeknik Pertanian Negeri Pangkep,3
18589,11213094,110293.0,Pelatihan <i>speaking</i> guru smk non bahasa ...,2018,Manajemen,76004,Nurhayati,Universitas Indraprasta PGRI Jakarta,3


In [24]:
artikelJurnalCategoryYear

,Nama Jurnal,Penerbit,Kategori,Tahun,Jumlah Artikel
744,Jurnal Indonesia,Rumah Guru dan Konsultan Pendidikan Yayasan Gl...,Pendidikan,2017,374
617,Jamba ilmiah : penelitian tindakan,Kelompok Kerja Pengawas Sekolah Agam (Sumatera...,Pendidikan,2017,318
1236,Jurnal pejuang pendidikan,Yayasan Global Bakti Asih,Pendidikan,2017,278
1356,Jurnal rumah guru : publikasi ilmiah bagi pend...,Yayasan Global Bakti Asih Rumah Guru dan Konsu...,Pendidikan,2017,251
439,Indonesian journal of chemistry,Universitas Gadjah Mada. Jurusan Kimia,Kimia,2019,119
...,...,...,...,...,...
769,Jurnal Pedagogika,FIP Universitas Negeri Gorontalo,Pendidikan,2019,1
770,Jurnal Pembangunan Manusia,Badan Penelitian dan Pengembangan Daerah Provi...,Pendidikan,2018,1
1392,Jurnal sistem dan informatika,Sekolah Tinggi Manajemen Informatika dan Tekni...,Komputer dan Teknologi Informasi,2017,1
1200,Jurnal masyarakat informatika,Program Studi Teknik Informatika Universitas D...,Komputer dan Teknologi Informasi,2017,1


In [16]:
import numpy as np

elon_list = [11, 21, 19, 18, 29]
elon_array = np.array(elon_list)

print(elon_array)
print(type(elon_array))

[11 21 19 18 29]
<class 'numpy.ndarray'>


In [20]:
for i in elon_array:
    print(i)

11
21
19
18
29
